# Enlace para compartir archivo

## Link notebook:
### https://drive.google.com/file/d/1cTRrzR052oyeDsC1nLHQYjyDnRUebgpD/view?usp=sharing
## Link presentacion:
### https://drive.google.com/file/d/1E9NH9EFPoze60LEtDHEqDD4hnOL_IbwS/view?usp=sharing

# Proyecto de pruebas A/B

### Objetivos del Estudio
* **Objetivo de Negocio:** Evaluar si la introducción de un sistema de recomendaciones mejorado incrementa la conversión de usuarios en el embudo de compra.
* **Objetivo Técnico:**
  * Comparar el comportamiento del grupo A (control) vs grupo B (nuevo embudo de pago) en 3 etapas clave.
  * Verificar un aumento mínimo del 10% en cada etapa del funnel:
    * product_page → visualizaciones de página de product
    * product_card → artículos añadidos al carrito
    * purchase → compras completadas
  * Validar que el experimento se ejecutó correctamente (sin contaminación de grupos, sin eventos de marketing confusos, con la muestra esperada de ~6,000 usuarios de la región UE).

### Tareas:

Explorar los datos:
* ¿Es necesario convertir los tipos?
* ¿Hay valores ausentes o duplicados? Si es así, ¿cómo los caracterizarías?

Llevar a cabo el análisis exploratorio de datos:
* Estudia la conversión en las diferentes etapas del embudo.
* ¿El número de eventos por usuario está distribuido equitativamente entre las muestras?
* ¿Hay usuarios que están presentes en ambas muestras?
* ¿Cómo se distribuye el número de eventos entre los días?
* ¿Hay alguna peculiaridad en los datos que hay que tener en cuenta antes de iniciar la prueba A/B?

Evaluar los resultados de la prueba A/B:
* ¿Qué puedes decir sobre los resultados de la prueba A/B?
*Utiliza una prueba z para comprobar la diferencia estadística entre las proporciones.

Describir conclusiones con respecto a la etapa EDA y los resultados de la prueba A/B.

## Paso 1. Carga de datos e Inspección Inicial

In [ ]:
# importo librerias
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import datetime
from statsmodels.stats.proportion import proportions_ztest


In [ ]:
# cargo los datasets
marketing   = pd.read_csv('/datasets/ab_project_marketing_events_us.csv')
new_users   = pd.read_csv('/datasets/final_ab_new_users_upd_us.csv')
events      = pd.read_csv('/datasets/final_ab_events_upd_us.csv')
participants = pd.read_csv('/datasets/final_ab_participants_upd_us.csv')

datasets = {
    'marketing':    marketing,
    'new_users':    new_users,
    'events':       events,
    'participants': participants
}

# realizo inspección inicial de los dataframe
# defino una función que me ayude en la inspección de los 4 dataframe
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  DATASET: {name.upper()}")
    print(f"{'='*60}")
    print(f"  Forma:   {df.shape[0]:,} filas × {df.shape[1]} columnas")
    print(f"\n--- Primeras filas ---")
    display(df.head(3))
    print(f"\n--- Tipos de datos e info general ---")
    df.info()

# ── VALORES AUSENTES ────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  VALORES AUSENTES")
print("="*60)

for name, df in datasets.items():
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(2)
    result  = pd.DataFrame({'missing': missing, '%': pct})
    result  = result[result['missing'] > 0]
    if result.empty:
        print(f"\n ok {name}: sin valores ausentes")
    else:
        print(f"\n Alerta  {name}:")
        print(result)

# ── DUPLICADOS ──────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  DUPLICADOS")
print("="*60)

for name, df in datasets.items():
    # Duplicados totales (filas idénticas)
    dup_full = df.duplicated().sum()
    print(f"\n{name}:")
    print(f"  Filas completamente duplicadas: {dup_full}")



### Resultado de la inspección inicial:

#### 1. ¿Es necesario convertir los tipos?
**Sí, es estrictamente necesario.**
Todas las columnas que contienen información cronológica han sido detectadas erróneamente como tipo object (cadenas de texto). Para poder realizar filtros temporales, calcular la ventana de conversión de 14 días y analizar la secuencia de eventos, debemos convertirlas al tipo adecuado.

**Columnas específicas a transformar a datetime:**
* Marketing Events: start_dt y finish_dt.
* New Users: first_date.Events: event_dt.

#### 2. ¿Hay valores ausentes o duplicados? Si es así, ¿cómo los caracterizarías?
* **Valores Duplicados:**
**Ninguno.** El análisis arroja 0 filas duplicadas exactas en los cuatro conjuntos de datos. Esto demuestra una excelente calidad de registro inicial en los logs individuales.
* **Valores Ausentes:**
**Únicamente en la tabla Events (columna details):** Faltan 363,447 valores (aproximadamente el 85.7% del dataset).
**Caracterización:** Estos valores ausentes son estructurales y normales (no representan un error técnico). El campo details almacena datos numéricos adicionales específicos del evento. En una tienda en línea, como se indica en la descripción de la estructura del archivo final_ab_events_upd_us.csv, este este valor suele ser el monto de la compra en dólares:

  ****details — datos adicionales sobre el evento (por ejemplo, el pedido total en USD para los eventos purchase)****

   Por lo tanto, solo el evento purchase genera este registro (lo que coincide con las 60,314 filas válidas). Eventos como login, product_page o product_cart no tienen un costo asociado y legítimamente deben permanecer vacíos. Se sugiere que No se deben eliminar.

## Hallazgos clave
* **Valores nulos en details son estructurales**
El 85.77% de nulos es completamente normal: solo los eventos de tipo purchase tienen monto en USD. Los eventos product_page y product_card no necesitan este campo.
* **Participantes: 14,525 vs 6,000 esperados**
La tabla de participantes tiene más del doble de los usuarios previstos. Es necesario verificar si hay usuarios fuera de la región EU, usuarios de otras pruebas A/B mezclados, o usuarios registrados fuera del período 7–21 dic.
Posible contaminación de grupos — pendiente por verificar

In [ ]:
# convierto tipos de datos a datetime
# Columnas que deben ser datetime
date_cols = {
    'marketing':    ['start_dt', 'finish_dt'],
    'new_users':    ['first_date'],
    'events':       ['event_dt'],
}

for name, cols in date_cols.items():
    df = datasets[name]
    for col in cols:
        df[col] = pd.to_datetime(df[col])
    print(f" OK {name}: columnas {cols} convertidas a datetime")

# verificación post-conversión
for name, cols in date_cols.items():
    print(f"\n{name} — tipos actualizados:")
    print(datasets[name][cols].dtypes)

## Paso 2: Análisis Exploratorio de Datos (EDA)

In [ ]:
# analizo con enfoque a lo que se necesita para conirmar la prueba antes de pasar a la limpieza y preparación de datos
# identifico duplicados lógicos importantes por dataset
# new_users: ¿un user_id puede aparecer más de una vez?
dup_users = new_users.duplicated(subset='user_id').sum()
print(f"\n  new_users — user_id duplicados:    {dup_users}")

# participants: ¿un usuario en más de un grupo/prueba?
dup_part = participants.duplicated(subset=['user_id', 'ab_test']).sum()
print(f"  participants — (user_id, ab_test) duplicados: {dup_part}")

# ¿Usuarios en ambos grupos A y B? (contaminación de grupos)
users_per_group = participants.groupby('user_id')['group'].nunique()
contaminated    = (users_per_group > 1).sum()
print(f"  participants — usuarios en AMBOS grupos (contaminación): {contaminated}")

# ── ESTADÍSTICAS DESCRIPTIVAS DE EVENTOS ───────────────────────────────────
print("\n" + "="*60)
print("  DISTRIBUCIÓN DE TIPOS DE EVENTO")
print("="*60)
print(events['event_name'].value_counts())

print("\n--- Rango de fechas por dataset ---")
print(f"  new_users  — first_date: {new_users['first_date'].min()} → {new_users['first_date'].max()}")
print(f"  events     — event_dt:   {events['event_dt'].min()} → {events['event_dt'].max()}")
print(f"  marketing  — campañas:   {marketing['start_dt'].min()} → {marketing['finish_dt'].max()}")

# ── PARTICIPANTES POR GRUPO ─────────────────────────────────────────────────
print("\n" + "="*60)
print("  PARTICIPANTES POR GRUPO")
print("="*60)
print(participants.groupby(['ab_test', 'group'])['user_id'].count())

# Verifico que solo existe la prueba de interés
print("\nPruebas únicas en participants:")
print(participants['ab_test'].unique())

### Hallazgos de EDA

* Distribución de eventos en el funnel
  * login:182,465 - 100%
  * product_page: 120,862- 66.2%
  * product_cart: 60,120- 33.0%
  * purchase: 60,314- 33.1%
* purchase (60,314) ≈ product_cart (60,120) — lógica rota
Hay casi tantas compras como adiciones al carrito. En un funnel normal, las compras deben ser un subconjunto del carrito. Esto sugiere eventos mal etiquetados, usuarios que compraron sin pasar por el carrito, o datos de usuarios pre-existentes mezclados.

#### Problema crítico 1 — Contaminación de grupos
* Usuarios en ambos grupos: 441 deben ser eliminados
* Grupo A (recomendador): 2,747, grupo de control
* Grupo B (recomendador): 928, sólo 34% de A

**Desbalance severo A (2,747) vs B (928)**
El grupo B tiene apenas el 34% de usuarios del grupo A. Para una prueba con 6,000 usuarios esperados y distribución 50/50, esto es una señal de reclutamiento fallido o filtrado incorrecto. El desbalance afecta la potencia estadística del test.

### Problema crítico 2 — Prueba contaminada con otro experimento
* interface_eu_test — Grupo A: 5,467 usuarios mezclados en la misma tabla de participantes **(Necesario Excluir)**
* interface_eu_test — Grupo B: 5,383 usuarios — total: 10,850 usuarios de otra prueba **(Ncesario Excluir)**

10,850 usuarios de interface_eu_test explican los 14,525 totales
La tabla participants mezcla dos pruebas A/B distintas. Al filtrar solo recommender_system_test quedan 3,675 usuarios — muy por debajo de los 6,000 esperados. Esto también explica la posible contaminación de los 441 usuarios: podrían estar en ambas pruebas simultáneamente.

### Problema crítico 3 — Fechas fuera de rango
* new_users — fecha máxima: 2020-12-23
Se registraron usuarios hasta el 23 dic, pero el cierre era el 21 dic **(Filtrar)**
* events — fecha máxima: 2020-12-30
La prueba terminaba el 2021-01-01, pero los eventos se cortan antes **(Verificar)**
* Campaña navideña activa
* Christmas & New Year Promo: 25 dic 2020 – 3 ene 2021 — solapa con la prueba **(Riesgo alto)**


## Paso 3: Limpieza y Preparación de Datos

In [ ]:
# ── FASE DE LIMPIEZA ──────────────────────────────────────────────────────────

# 1. Filtro solo la prueba correcta
reco_participants = participants[
    participants['ab_test'] == 'recommender_system_test'
].copy()
print(f"Participantes de recommender_system_test: {len(reco_participants)}")

# 2. Elimino usuarios contaminados (en ambos grupos)
contaminated_users = (
    reco_participants.groupby('user_id')['group']
    .nunique()
    .pipe(lambda s: s[s > 1].index)
)
reco_clean = reco_participants[~reco_participants['user_id'].isin(contaminated_users)]
print(f"Tras eliminar contaminados: {len(reco_clean)}")

# 3. Filtro new_users: solo región EU y período correcto (7–21 dic)
eu_new_users = new_users[
    (new_users['region'] == 'EU') &
    (new_users['first_date'] >= '2020-12-07') &
    (new_users['first_date'] <= '2020-12-21')
]
print(f"Nuevos usuarios EU en período: {len(eu_new_users)}")

# 4. Solo participantes que sean nuevos usuarios EU válidos
valid_ids = set(reco_clean['user_id']) & set(eu_new_users['user_id'])
reco_final = reco_clean[reco_clean['user_id'].isin(valid_ids)]
print(f"Participantes válidos finales: {len(reco_final)}")
print(reco_final['group'].value_counts())

# 5. Filtro eventos: solo usuarios válidos, dentro de 14 días desde registro
# (uno con fecha de registro para calcular días transcurridos)
events_clean = events[events['user_id'].isin(valid_ids)].copy()

events_clean['event_dt'] = pd.to_datetime(events_clean['event_dt'])
eu_new_users = eu_new_users.copy()
eu_new_users['first_date'] = pd.to_datetime(eu_new_users['first_date'])

events_clean = events_clean.merge(
    eu_new_users[['user_id', 'first_date']], on='user_id', how='left'
)
events_clean['days_since_reg'] = (
    events_clean['event_dt'] - events_clean['first_date']
).dt.days
events_clean = events_clean[
    (events_clean['days_since_reg'] >= 0) &
    (events_clean['days_since_reg'] <= 14)
]
print(f"\nEventos limpios (14 días desde registro): {len(events_clean)}")
print(events_clean['event_name'].value_counts())

# 6. Reviso solapamiento con campaña navideña
christmas_start = pd.Timestamp('2020-12-25')
events_with_christmas = events_clean[events_clean['event_dt'] >= christmas_start]
print(f"\nEventos durante campaña navideña: {len(events_with_christmas)}")
print(f"% del total: {len(events_with_christmas)/len(events_clean)*100:.1f}%")

In [ ]:
# ── FASE DE PREPARACIÓN DE DATOS ──────────────────────────────────────────────────────────
# ── CREAR events_with_group ─────────────────────────────────────────────────
events_with_group = events_clean.merge(
    reco_final[['user_id', 'group']], on='user_id', how='inner'
)
events_with_group['date'] = events_with_group['event_dt'].dt.date

# ── 1. EVENTOS POR USUARIO — distribución por grupo ───────────────────────────
events_per_user = (
    events_with_group.groupby(['user_id', 'group'])
    .size()
    .reset_index(name='n_events')
)

print("=== Eventos por usuario por grupo ===")
print(events_per_user.groupby('group')['n_events']
      .describe().round(2))

# ── 2. DISTRIBUCIÓN DE EVENTOS POR DÍA ───────────────────────────────────────
events_with_group['date'] = events_with_group['event_dt'].dt.date

events_by_day = (
    events_with_group.groupby(['date', 'group'])
    .size()
    .reset_index(name='n_events')
)

print("\n=== Eventos por día (primeras y últimas fechas) ===")
print(events_by_day.pivot(index='date', columns='group', values='n_events')
      .fillna(0).astype(int))

# ── 3. CONVERSIÓN POR ETAPA DEL FUNNEL + TEST Z───────────────────────────────────────
funnel_events = ['product_page', 'product_cart', 'purchase']

user_funnel = (
    events_with_group[events_with_group['event_name'].isin(funnel_events)]
    .groupby(['user_id', 'group', 'event_name'])
    .size().reset_index(name='count')
    .assign(did_event=1)
    .pivot_table(index=['user_id','group'], columns='event_name',
                 values='did_event', fill_value=0)
    .reset_index()
)

totals = reco_final['group'].value_counts().to_dict()

print("\n=== Tasas de conversión y test z ===")
print(f"{'Evento':<15} {'A_n':>6} {'B_n':>6} {'A%':>8} {'B%':>8} {'lift':>8} {'p-value':>10} {'sig':>5}")
print("-" * 70)

results = {}
for event in funnel_events:
    if event not in user_funnel.columns:
        print(f"{event:<15} — columna no encontrada")
        continue
    a_conv = user_funnel[user_funnel['group']=='A'][event].sum()
    b_conv = user_funnel[user_funnel['group']=='B'][event].sum()
    a_rate = a_conv / totals['A']
    b_rate = b_conv / totals['B']
    lift   = (b_rate - a_rate) / a_rate * 100
    z, p   = proportions_ztest(
        [b_conv, a_conv], [totals['B'], totals['A']]
    )
    results[event] = dict(a_conv=a_conv, b_conv=b_conv,
                          a_rate=a_rate, b_rate=b_rate,
                          lift=lift, p=p, sig=(p < 0.05))
    print(f"{event:<15} {int(a_conv):>6} {int(b_conv):>6} "
          f"{a_rate:>7.1%} {b_rate:>7.1%} {lift:>+7.1f}% "
          f"{p:>10.4f} {'OK' if p < 0.05 else 'X':>5}")



In [ ]:
# ── 4. VISUALIZACIONES ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# — Funnel comparativo A vs B —
ax1 = fig.add_subplot(gs[0, 0])
stages  = [e.replace('product_', 'prod_') for e in funnel_events]
rates_a = [results[e]['a_rate'] for e in funnel_events]
rates_b = [results[e]['b_rate'] for e in funnel_events]
x = np.arange(len(stages)); w = 0.35
ax1.bar(x - w/2, rates_a, w, label='Grupo A', color='steelblue', alpha=0.85)
ax1.bar(x + w/2, rates_b, w, label='Grupo B', color='coral',     alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(stages)
ax1.set_ylabel('Tasa de conversión')
ax1.set_title('Conversión por etapa del funnel')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax1.legend(); ax1.grid(axis='y', alpha=0.3)
for i, (ra, rb) in enumerate(zip(rates_a, rates_b)):
    ax1.text(i - w/2, ra + 0.003, f'{ra:.1%}', ha='center', fontsize=8)
    ax1.text(i + w/2, rb + 0.003, f'{rb:.1%}', ha='center', fontsize=8)

# — Lift por etapa —
ax2 = fig.add_subplot(gs[0, 1])
lifts  = [results[e]['lift'] for e in funnel_events]
colors = ['green' if l >= 10 else ('orange' if l >= 0 else 'red') for l in lifts]
ax2.bar(stages, lifts, color=colors, alpha=0.8)
ax2.axhline(10, color='green', linestyle='--', linewidth=1.2, label='Objetivo +10%')
ax2.axhline(0,  color='gray',  linestyle='-',  linewidth=0.8)
ax2.set_ylabel('Lift (%)'); ax2.set_title('Lift B vs A por etapa')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
for i, l in enumerate(lifts):
    ax2.text(i, l + 0.5, f'{l:+.1f}%', ha='center', fontsize=9)

# — Eventos por día —
ax3 = fig.add_subplot(gs[1, :])
pivot = (events_by_day
         .pivot(index='date', columns='group', values='n_events')
         .fillna(0))
dates  = list(pivot.index)
x_pos  = range(len(dates))
for group, color in [('A', 'steelblue'), ('B', 'coral')]:
    if group in pivot.columns:
        ax3.plot(x_pos, pivot[group].values,
                 color=color, marker='o', markersize=3,
                 linewidth=1.5, label=f'Grupo {group}')

# Marcar inicio de campaña navideña
christmas = datetime.date(2020, 12, 25)
if christmas in dates:
    xmas_idx = dates.index(christmas)
    ax3.axvspan(xmas_idx, len(dates), color='orange',
                alpha=0.1, label='Campaña navideña')
    ax3.axvline(xmas_idx, color='orange',
                linestyle='--', linewidth=1, alpha=0.7)

ax3.set_xticks(list(x_pos))
ax3.set_xticklabels([str(d) for d in dates], rotation=45, ha='right')
ax3.set_title('Eventos por día por grupo')
ax3.set_ylabel('Número de eventos')
ax3.legend(); ax3.grid(alpha=0.3)

plt.suptitle('Análisis A/B — recommender_system_test', fontsize=14, fontweight='bold')
plt.savefig('ab_test_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Gráfica guardada como ab_test_analysis.png")

## Resultados del test z por etapa del funnel
### Veredicto: la prueba A/B falló en los 3 objetivos
* **El grupo B convierte PEOR que el grupo A en todas las etapas**
El nuevo sistema de recomendaciones no solo no mejoró la conversión — la empeoró significativamente. En lugar de un +10% en cada etapa, el grupo B muestra caídas de −13.1%, −7.4% y −11.2%. El objetivo no fue alcanzado en ninguna de las tres etapas del funnel.
* **product_cart: única etapa sin significancia estadística:**
Con p=0.2147, la diferencia en la etapa de carrito no es estadísticamente significativa (p > 0.05). No podemos afirmar con confianza que el grupo B sea peor en esta etapa — la diferencia podría ser ruido. Las otras dos etapas sí muestran diferencias reales (p < 0.05).

### Distribución de eventos por usuario
* Media eventos — Grupo A: 6.67, mediana: 6.0
* Media eventos — Grupo B: 5.45, mediana: 4.0
* Diferencia en media: −18.3%, B genera menos actividad

* **Los usuarios del grupo B generan significativamente menos eventos**
Media de 6.67 vs 5.45 eventos por usuario, y mediana de 6.0 vs 4.0. Esto refuerza que el nuevo funnel reduce el engagement en lugar de aumentarlo. La distribución no es equitativa entre grupos — B tiene menor actividad en todas las métricas.

### Anomalías en distribución temporal de eventos
* Crecimiento acelerado grupo A desde el 14 dic: A pasa de ~330 eventos/día a 1,015 el 14 dic y sigue subiendo hasta 1,903 el 21 dic. B se mantiene relativamente estable entre 160–400. **Anomalía**
* 25 de diciembre ausente en los datos: No hay registro del 25 dic (día de Navidad) — salta del 24 al 26. Posible fallo en recolección de datos ese día.**Verificar**
* Caída abrupta después del 21 dic en grupo A: Del pico de 1,903 el 21 dic cae a 1,217 el 22 dic. Coincide con el cierre de admisión de nuevos usuarios.**Esperado**
* Ratio A/B se dispara desde el 14 dic: Del 7 al 13 dic el ratio A:B es cercano a 1:1. A partir del 14 dic el grupo A domina con ratios de 4:1 o más. Esto es inconsistente con el desbalance 3:1 en usuarios.**Anomalía crítica**


## Conclusiones EDA
1. Desbalance grave en los grupos del experimento
Tras la limpieza, el grupo A quedó con 2,604 usuarios y el grupo B con solo 877 — un ratio de 3:1 cuando se esperaba 1:1. Esto no fue intencional: las especificaciones indicaban ~6,000 participantes totales. El reclutamiento del grupo B fue claramente deficiente, lo que compromete la potencia estadística del test.
2. Dos pruebas A/B corriendo simultáneamente sobre la misma audiencia
La tabla de participantes mezclaba recommender_system_test e interface_eu_test, con 10,850 usuarios de la segunda prueba. Esto introduce un efecto confundidor: los usuarios expuestos a ambos experimentos al mismo tiempo no permiten aislar el impacto de cada cambio.
3. Campaña navideña solapada con el período de la prueba
La campaña Christmas & New Year Promo estuvo activa desde el 25 de diciembre, representando el 9.2% de los eventos limpios. Si esta campaña afectó de forma diferente a los grupos A y B, los resultados de conversión estarían parcialmente explicados por el efecto estacional y no por el sistema de recomendaciones.
4. Anomalía en la distribución temporal de eventos
Del 7 al 13 de diciembre, el ratio de eventos diarios entre A y B era cercano a 1:1. A partir del 14 de diciembre, el grupo A dispara su actividad hasta 1,903 eventos el 21 de diciembre, mientras el grupo B se mantiene estable entre 160–400. Esta divergencia es inconsistente con el simple desbalance en número de usuarios y sugiere un factor externo no controlado.
5. purchase supera a product_cart en el funnel
Tanto en el dataset global como tras la limpieza, hay más eventos de compra que de adición al carrito (3,018 vs 2,935). En un funnel bien instrumentado, toda compra debería estar precedida por un evento de carrito. Esto apunta a un problema de tracking: el evento product_cart no se está registrando en todos los flujos de compra
6. Sin duplicados ni contaminación real dentro del experimento
Los 441 usuarios que parecían estar en ambos grupos pertenecían exclusivamente a interface_eu_test. Dentro de recommender_system_test no hay contaminación: ningún usuario aparece en A y B simultáneamente. Los datos no tienen filas duplicadas ni valores ausentes problemáticos.
7. Los usuarios del grupo B generan menos actividad por persona
Media de eventos por usuario: A = 6.67, B = 5.45 (mediana: 6 vs 4). El grupo B no solo tiene menos usuarios — cada usuario interactúa menos con la plataforma. Esto es coherente con los peores resultados de conversión, pero también puede reflejar un sesgo en qué tipo de usuarios fueron asignados a cada grupo.

## Conclusiones prueba A/B
### Veredicto: la prueba A/B no puede considerarse válida ni exitosa
El nuevo sistema de recomendaciones no alcanzó el objetivo de +10% en ninguna de las tres etapas del funnel.** Por el contrario, el grupo B mostró peores tasas de conversión en todas las etapas. Sin embargo, las condiciones del experimento impiden atribuir estos resultados únicamente al sistema probado.
#### Ninguna etapa alcanzó el objetivo de +10% de lift
product_page: lift = −13.1% (p < 0.0001) — estadísticamente significativo, B peor que A.
product_cart: lift = −7.4% (p = 0.2147) — no significativo, diferencia no concluyente.
purchase: lift = −11.2% (p = 0.0465) — estadísticamente significativo, B peor que A.
#### Baja potencia estadística por insuficiencia del grupo B
Con solo 877 usuarios en B frente a los ~3,000 esperados, el test tiene menor potencia para detectar diferencias reales. En particular, la no significancia en product_cart (p=0.2147) podría deberse al tamaño insuficiente de la muestra, no a la ausencia de efecto.

### Recomendación: no lanzar el sistema de recomendaciones
Los resultados no justifican el lanzamiento. Se recomienda repetir el experimento corrigiendo: (1) asignación aleatoria balanceada al 50/50, (2) aislamiento de otras pruebas A/B concurrentes, (3) evitar solapamiento con campañas de marketing, y (4) corregir el tracking de product_cart antes de volver a medir.